# Financial Forecasting and Valuation Workflow

This notebook is a public reconstruction of the valuation logic behind the finance project. The original Excel workbook was not found in the local scan, so the notebook focuses on transparent DCF, WACC, CAPM, NPV, IRR, and sensitivity logic.

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd

In [ ]:
forecast = pd.DataFrame({
    "year": [2023, 2024, 2025, 2026, 2027],
    "revenue": [1200, 1320, 1452, 1583, 1710],
    "ebit_margin": [0.14, 0.15, 0.155, 0.16, 0.165],
    "tax_rate": [0.24, 0.24, 0.24, 0.24, 0.24],
    "depreciation_pct_revenue": [0.035, 0.035, 0.034, 0.034, 0.033],
    "capex_pct_revenue": [0.055, 0.054, 0.052, 0.051, 0.050],
    "nwc_pct_revenue_change": [0.08, 0.08, 0.075, 0.075, 0.07],
})
forecast

In [ ]:
risk_free_rate = 0.042
market_risk_premium = 0.055
beta = 1.15
pre_tax_cost_of_debt = 0.065
tax_rate = 0.24
equity_weight = 0.72
debt_weight = 0.28

cost_of_equity = risk_free_rate + beta * market_risk_premium
after_tax_cost_of_debt = pre_tax_cost_of_debt * (1 - tax_rate)
wacc = equity_weight * cost_of_equity + debt_weight * after_tax_cost_of_debt
wacc

In [ ]:
forecast["ebit"] = forecast["revenue"] * forecast["ebit_margin"]
forecast["nopat"] = forecast["ebit"] * (1 - forecast["tax_rate"])
forecast["depreciation"] = forecast["revenue"] * forecast["depreciation_pct_revenue"]
forecast["capex"] = forecast["revenue"] * forecast["capex_pct_revenue"]
forecast["revenue_change"] = forecast["revenue"].diff().fillna(forecast["revenue"].iloc[0] * 0.10)
forecast["change_nwc"] = forecast["revenue_change"] * forecast["nwc_pct_revenue_change"]
forecast["free_cash_flow"] = forecast["nopat"] + forecast["depreciation"] - forecast["capex"] - forecast["change_nwc"]
forecast.round(2)

In [ ]:
def dcf_value(cash_flows: pd.Series, discount_rate: float, terminal_growth: float) -> float:
    periods = np.arange(1, len(cash_flows) + 1)
    pv_fcf = (cash_flows / (1 + discount_rate) ** periods).sum()
    terminal_value = cash_flows.iloc[-1] * (1 + terminal_growth) / (discount_rate - terminal_growth)
    pv_terminal = terminal_value / (1 + discount_rate) ** len(cash_flows)
    return float(pv_fcf + pv_terminal)

enterprise_value = dcf_value(forecast["free_cash_flow"], wacc, terminal_growth=0.025)
enterprise_value

In [ ]:
sensitivity = []
for discount_rate in np.arange(wacc - 0.02, wacc + 0.021, 0.01):
    for terminal_growth in [0.015, 0.020, 0.025, 0.030, 0.035]:
        sensitivity.append({
            "discount_rate": round(discount_rate, 4),
            "terminal_growth": terminal_growth,
            "enterprise_value": dcf_value(forecast["free_cash_flow"], discount_rate, terminal_growth),
        })

pd.DataFrame(sensitivity).pivot(index="discount_rate", columns="terminal_growth", values="enterprise_value").round(1)

## Notes For Reviewers

The assumptions above are illustrative placeholders. The purpose of this notebook is to make the model mechanics reviewable in GitHub while the original Excel workbook remains unavailable.